# 4장. pandas로 데이터에 질문하기

이 노트북은 데이터를 선택·필터링·정렬하고, 안전하게 병합한 뒤 완료 주문 기준 요약표를 만드는 실습 자료입니다.


## 학습 목표

- 실제 컬럼명과 값의 종류를 확인한 뒤 필터링합니다.
- 수량과 단가로 주문 상세 금액을 계산합니다.
- `validate`와 `indicator`를 사용해 병합 결과를 검증합니다.
- 취소·환불 주문을 제외하고 완료 주문 기준 매출을 계산합니다.
- 개인정보를 최소화한 분석 결과를 저장합니다.


## 1. 프로젝트 루트와 실행 환경 확인

VS Code에서 Notebook을 실행하면 현재 작업 폴더가 프로젝트 루트 또는 `notebooks` 폴더일 수 있습니다. 아래 코드는 상위 폴더를 확인해 프로젝트 루트를 찾습니다.


In [162]:
from pathlib import Path
import sys

import pandas as pd


def find_project_root(start_path):
    start_path = Path(start_path).resolve()
    for candidate in [start_path, *start_path.parents]:
        if (candidate / 'requirements.txt').exists() and (candidate / 'scripts').exists():
            return candidate
    raise FileNotFoundError('프로젝트 루트 폴더를 찾을 수 없습니다.')


PROJECT_ROOT = find_project_root(Path.cwd())
DATA_DIR = PROJECT_ROOT / 'data' / 'raw'
REPORT_DIR = PROJECT_ROOT / 'reports'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print('Python 실행 파일:', sys.executable)
print('현재 작업 폴더:', Path.cwd())
print('프로젝트 루트:', PROJECT_ROOT)
print('데이터 폴더:', DATA_DIR)
print('결과 저장 폴더:', REPORT_DIR)


Python 실행 파일: d:\git\LLM\llm-data-analysis-course\.venv\Scripts\python.exe
현재 작업 폴더: d:\git\LLM\llm-data-analysis-course\practice\chapter04
프로젝트 루트: D:\git\LLM\llm-data-analysis-course
데이터 폴더: D:\git\LLM\llm-data-analysis-course\data\raw
결과 저장 폴더: D:\git\LLM\llm-data-analysis-course\reports


## 2. 데이터 파일 확인과 불러오기

파일이 없다면 프로젝트 루트에서 `python scripts/generate_sample_data.py`를 먼저 실행하세요.


In [163]:
required_files = ['customers.csv', 'products.csv', 'orders.csv', 'order_items.csv']
missing_files = [name for name in required_files if not (DATA_DIR / name).exists()]

if missing_files:
    raise FileNotFoundError(
        '필요한 데이터 파일이 없습니다: ' + ', '.join(missing_files)
        + '. 프로젝트 루트에서 python scripts/generate_sample_data.py를 실행하세요.'
    )

customers = pd.read_csv(DATA_DIR / 'customers.csv')
products = pd.read_csv(DATA_DIR / 'products.csv')
orders = pd.read_csv(DATA_DIR / 'orders.csv')
order_items = pd.read_csv(DATA_DIR / 'order_items.csv')

print('데이터 불러오기 완료')


데이터 불러오기 완료


## 3. 데이터 구조와 필수 컬럼 확인

LLM이 작성한 코드가 실제 컬럼명과 일치하는지 먼저 확인합니다.


In [18]:
datasets = {
    'customers': customers,
    'products': products,
    'orders': orders,
    'order_items': order_items,
}

expected_columns = {
    'customers': ['customer_id', 'gender', 'age', 'city'],
    'products': ['product_id', 'product_name', 'category', 'price'],
    'orders': ['order_id', 'customer_id', 'order_date', 'order_status'],
    'order_items': ['order_id', 'product_id', 'quantity', 'unit_price'],
}

for name, df in datasets.items():
    missing = [col for col in expected_columns[name] if col not in df.columns]
    print(name, df.shape, df.columns.tolist())
    if missing:
        raise KeyError(f'{name}에 필요한 컬럼이 없습니다: {missing}')


customers (150, 6) ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']
products (100, 4) ['product_id', 'product_name', 'category', 'price']
orders (300, 5) ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status']
order_items (764, 5) ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price']


## 4. 컬럼 선택과 행 필터링

이 저장소의 샘플 데이터는 도시명을 `서울`, `부산`처럼 한글로 생성합니다. 필터링 전에 실제 값을 확인합니다.


In [167]:
customer_basic = customers[['customer_id', 'gender', 'age', 'city']]
display(customer_basic.head())

print('도시별 고객 수')
display(customers['city'].value_counts())

customers_over_30 = customers[customers['age'] >= 30]
seoul_customers = customers[customers['city'] == '서울']
city_customers = customers[customers['city'].isin(['서울', '부산'])]

print('30세 이상 고객 수:', len(customers_over_30))
print('서울 고객 수:', len(seoul_customers))
print('서울 또는 부산 고객 수:', len(city_customers))


,customer_id,gender,age,city
0,1,F,19,광주
1,2,F,32,대구
2,3,F,61,성남
3,4,F,55,울산
4,5,F,19,부산


도시별 고객 수


city
성남    21
광주    17
부산    16
대구    15
서울    15
울산    14
인천    14
대전    14
수원    13
고양    11
Name: count, dtype: int64

30세 이상 고객 수: 111
서울 고객 수: 15
서울 또는 부산 고객 수: 31


In [ ]:
#필요한 컬럼만 선택합니다.
customer_basic = customers[[
    "customer_id",
    "gender",
    "age",
    "city",
]]

#30세 이상 고객을 선택합니다.
customers_over_30 = customers[
    customers["age"] >= 30
]

#서울 또는 부산 고객을 선택합니다.
city_customers = customers[
    customers["city"].isin(["서울", "부산"])
]

#필터링 전에 실제 값을 확인합니다.
customers["city"].value_counts()
orders["order_status"].value_counts()



order_status
completed    184
cancelled     64
refunded      52
Name: count, dtype: int64

## 5. 정렬과 주문 상태 확인


In [168]:
display(products.sort_values('price', ascending=False).head(10))
display(customers.sort_values('age', ascending=False).head(10))

print('주문 상태별 건수')
display(orders['order_status'].value_counts())


,product_id,product_name,category,price
98,99,뷰티 상품 099,뷰티,200000
69,70,패션 상품 070,패션,198000
57,58,식품 상품 058,식품,197000
42,43,뷰티 상품 043,뷰티,197000
23,24,스포츠 상품 024,스포츠,196000
8,9,스포츠 상품 009,스포츠,193000
36,37,뷰티 상품 037,뷰티,193000
71,72,뷰티 상품 072,뷰티,189000
7,8,스포츠 상품 008,스포츠,189000
52,53,생활용품 상품 053,생활용품,188000


,customer_id,name,gender,age,city,signup_date
14,15,장정식,M,69,서울,2026-08-11
8,9,송지민,M,69,서울,2025-12-26
54,55,윤영철,M,69,광주,2023-10-07
78,79,이서준,M,69,부산,2023-09-20
136,137,김선영,M,68,울산,2025-02-04
45,46,김서현,M,67,대전,2026-07-20
132,133,김성진,M,67,성남,2026-08-05
123,124,김시우,M,67,대구,2024-12-01
11,12,김정남,F,66,대전,2024-12-07
57,58,김성훈,F,66,성남,2025-08-21


주문 상태별 건수


order_status
completed    184
cancelled     64
refunded      52
Name: count, dtype: int64

## 6. 주문 상세 금액 만들기

`line_total`은 주문 상세 1행의 금액입니다. 주문 상태를 연결하기 전 합계는 확정 매출이 아니라 전체 주문 상세 금액입니다.


In [171]:
order_items = order_items.copy()
order_items['line_total'] = order_items['quantity'] * order_items['unit_price'] #`order_items`에 `line_total` 컬럼이 추가됩니다.

all_order_amount = order_items['line_total'].sum()
print('전체 주문 상세 금액:', all_order_amount)
display(order_items[['order_id', 'product_id', 'quantity', 'unit_price', 'line_total']].head())


전체 주문 상세 금액: 255610000


,order_id,product_id,quantity,unit_price,line_total
0,1,100,3,102000,306000
1,1,87,5,25000,125000
2,1,7,3,142000,426000
3,1,9,3,193000,579000
4,2,72,4,189000,756000


## 7. 주문 데이터 병합과 검증

`validate='many_to_one'`은 주문 상세의 동일 주문 ID는 여러 번 나올 수 있지만 주문 테이블의 주문 ID는 한 번만 나와야 한다는 뜻입니다.


In [174]:
print('orders.order_id 중복 수:', orders['order_id'].duplicated().sum()) #먼저 `orders.order_id`가 고유한지 확인합니다.

order_sales = order_items.merge( #그다음 병합합니다.
    orders[['order_id', 'customer_id', 'order_date', 'order_status']],
    on='order_id',
    how='left',
    validate='many_to_one',
    indicator=True,
)

 #병합 직후 확인합니다.
print('병합 전 행 수:', len(order_items))
print('병합 후 행 수:', len(order_sales))
display(order_sales['_merge'].value_counts())

order_sales = order_sales.drop(columns='_merge')


orders.order_id 중복 수: 0
병합 전 행 수: 764
병합 후 행 수: 764


_merge
both          764
left_only       0
right_only      0
Name: count, dtype: int64

## 8. 완료 주문만 선택하기

이후의 매출 요약은 `order_status == 'completed'`인 주문만 사용합니다.


In [177]:
#날짜 변환
order_sales['order_date'] = pd.to_datetime(order_sales['order_date'], errors='coerce')
print('날짜 변환 실패:', order_sales['order_date'].isna().sum())

#완료 주문만 선택합니다.
completed_order_sales = order_sales[
    order_sales['order_status'] == 'completed'
].copy()

#월 컬럼을 만듭니다.
completed_order_sales['order_month'] = (
    completed_order_sales['order_date'].dt.to_period('M').astype(str) 
)

print('전체 주문 상세 행 수:', len(order_sales))
print('완료 주문 상세 행 수:', len(completed_order_sales))
print('완료 주문 매출:', completed_order_sales['line_total'].sum())


날짜 변환 실패: 0
전체 주문 상세 행 수: 764
완료 주문 상세 행 수: 474
완료 주문 매출: 148990000


## 9. 상품 데이터 병합과 검증


In [178]:
print('products.product_id 중복 수:', products['product_id'].duplicated().sum())

completed_sales_items = completed_order_sales.merge(
    products,
    on='product_id',
    how='left',
    validate='many_to_one',
    indicator=True,
)

print('병합 전 행 수:', len(completed_order_sales))
print('병합 후 행 수:', len(completed_sales_items))
display(completed_sales_items['_merge'].value_counts()) #완료 주문이 0건이라면 먼저 실제 주문 상태를 확인합니다.

completed_sales_items = completed_sales_items.drop(columns='_merge')


products.product_id 중복 수: 0
병합 전 행 수: 474
병합 후 행 수: 474


_merge
both          474
left_only       0
right_only      0
Name: count, dtype: int64

## 10. 카테고리별·상품별 매출


In [179]:
category_sales = (
    completed_sales_items
    .groupby('category', as_index=False)
    .agg(
        total_quantity=('quantity', 'sum'),
        total_sales=('line_total', 'sum'),
    )
    .sort_values('total_sales', ascending=False)
)

category_sales['sales_ratio'] = (
    category_sales['total_sales'] / category_sales['total_sales'].sum() * 100
).round(2)

display(category_sales)

product_sales = (
    completed_sales_items
    .groupby(['product_id', 'product_name', 'category'], as_index=False)
    .agg(
        total_quantity=('quantity', 'sum'),
        total_sales=('line_total', 'sum'),
    )
    .sort_values('total_sales', ascending=False)
)

display(product_sales.head(10))


,category,total_quantity,total_sales,sales_ratio
3,스포츠,295,31743000,21.31
5,전자기기,259,26400000,17.72
2,생활용품,272,23915000,16.05
1,뷰티,223,23383000,15.69
4,식품,133,16573000,11.12
0,도서,149,16389000,11.00
6,패션,111,10587000,7.11


,product_id,product_name,category,total_quantity,total_sales
39,41,스포츠 상품 041,스포츠,35,5705000
11,12,식품 상품 012,식품,25,4375000
8,9,스포츠 상품 009,스포츠,20,3860000
70,72,뷰티 상품 072,뷰티,20,3780000
69,71,전자기기 상품 071,전자기기,23,3703000
66,68,스포츠 상품 068,스포츠,26,3640000
78,81,전자기기 상품 081,전자기기,22,3630000
10,11,패션 상품 011,패션,31,3565000
20,22,생활용품 상품 022,생활용품,29,3248000
86,89,생활용품 상품 089,생활용품,30,3090000


## 11. 월별 매출


In [ ]:
monthly_summary = (
    completed_order_sales
    .groupby('order_month', as_index=False)
    .agg(
        total_sales=('line_total', 'sum'),
        order_count=('order_id', 'nunique'),
    )
    .sort_values('order_month')
)

monthly_summary['average_order_value'] = (
    monthly_summary['total_sales'] / monthly_summary['order_count']
).round(0)

display(monthly_summary)


,order_month,total_sales,order_count,average_order_value
0,2025-09,2650000,5,530000.0
1,2025-10,16970000,19,893158.0
2,2025-11,11021000,13,847769.0
3,2025-12,21085000,23,916739.0
4,2026-01,14272000,16,892000.0
5,2026-02,4784000,7,683429.0
6,2026-03,18323000,25,732920.0
7,2026-04,11546000,16,721625.0
8,2026-05,16631000,19,875316.0
9,2026-06,12504000,17,735529.0


## 12. 고객별 구매 금액

고객 ID로 먼저 집계한 뒤 고객 속성을 붙입니다. 출력에는 실제 이름 대신 익명화된 고객 라벨을 사용합니다.


In [181]:
customer_sales = (
    completed_order_sales
    .groupby('customer_id', as_index=False)
    .agg(
        order_count=('order_id', 'nunique'),
        total_sales=('line_total', 'sum'),
    )
    .sort_values('total_sales', ascending=False)
)

customer_sales = customer_sales.merge(
    customers[['customer_id', 'city']],
    on='customer_id',
    how='left',
    validate='one_to_one',
)

customer_sales['customer_label'] = 'Customer ' + customer_sales['customer_id'].astype(str)
display(customer_sales[['customer_label', 'city', 'order_count', 'total_sales']].head(10))


,customer_label,city,order_count,total_sales
0,Customer 117,성남,5,4100000
1,Customer 102,고양,4,3996000
2,Customer 83,수원,4,3880000
3,Customer 30,서울,5,3590000
4,Customer 40,서울,4,3523000
5,Customer 20,인천,2,3191000
6,Customer 3,성남,2,3178000
7,Customer 111,광주,3,3153000
8,Customer 66,서울,4,3093000
9,Customer 147,부산,2,2990000


## 13. 결과 저장하기


In [182]:
outputs = {
    'ch04_category_sales.csv': category_sales,
    'ch04_product_sales.csv': product_sales,
    'ch04_monthly_sales.csv': monthly_summary,
    'ch04_customer_sales.csv': customer_sales,
}

for filename, df in outputs.items():
    path = REPORT_DIR / filename
    df.to_csv(path, index=False, encoding='utf-8-sig')
    print(filename, path.exists(), path.stat().st_size)


ch04_category_sales.csv True 254
ch04_product_sales.csv True 4409
ch04_monthly_sales.csv True 443
ch04_customer_sales.csv True 3387


## 14. LLM 코드 검증 연습

```text
LLM이 다음 코드를 제안했습니다.

category_sales = order_items.groupby('category')['line_total'].sum()

현재 데이터에서 이 코드가 바로 실행 가능한지 검토해 주세요.
category 컬럼의 위치, 주문 상태 필터링, merge validate와 indicator를 포함해
안전한 수정 코드와 검증 순서를 설명해 주세요.
```


## 15. 실습 과제

1. 40세 이상 고객을 추출합니다.
2. 상품 가격이 낮은 순서대로 10개를 출력합니다.
3. 결제수단별 주문 수와 비율을 계산합니다.
4. 카테고리별 평균 상품 가격을 계산합니다.
5. 완료 주문과 전체 주문의 금액 차이를 계산합니다.
6. 병합 검증을 포함한 고객별 구매 금액 코드 요청 프롬프트를 작성합니다.


##### 1) 40세 이상 고객 추출

In [183]:
customers_over_40 = customers[
    customers["age"] >= 40
]

display(customers_over_40)


,customer_id,name,gender,age,city,signup_date
2,3,이경수,F,61,성남,2024-08-19
3,4,조영호,F,55,울산,2026-06-20
6,7,이상현,F,53,인천,2025-02-18
8,9,송지민,M,69,서울,2025-12-26
9,10,유도현,F,62,울산,2024-12-13
...,...,...,...,...,...,...
140,141,김진호,M,40,광주,2023-11-16
142,143,김예은,F,43,대전,2024-03-10
144,145,조정호,M,59,성남,2024-03-10
145,146,김숙자,M,61,성남,2026-03-01


##### 2) 상품 가격이 낮은 순서대로 10개

최저 가격은 5,000원이었으며, 해당 가격의 상품이 2개 존재함

In [184]:
products_low_price = products.sort_values(
    "price",ascending=True #오름차순으로 정렬
).head(10) #앞에 10개 자름

display(products_low_price)

,product_id,product_name,category,price
5,6,전자기기 상품 006,전자기기,5000
27,28,뷰티 상품 028,뷰티,5000
97,98,스포츠 상품 098,스포츠,10000
46,47,생활용품 상품 047,생활용품,11000
94,95,전자기기 상품 095,전자기기,20000
49,50,뷰티 상품 050,뷰티,23000
82,83,전자기기 상품 083,전자기기,24000
86,87,도서 상품 087,도서,25000
41,42,패션 상품 042,패션,28000
58,59,뷰티 상품 059,뷰티,32000


##### 3) 결제수단별 주문 수와 비율

총 300건의 주문을 결제수단별로 확인한 결과, 
카카오페이가 79건(26.33%)으로 가장 많았고 카드가 70건(23.33%)으로 가장 적었다.

In [186]:
print(orders.columns.tolist()) #실제 컬럼 확인!!!!

payment_count = orders["payment_method"].value_counts()
payment_ratio = (
    orders["payment_method"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("결제수단별 주문 수")
display(payment_count)

print("결제수단별 비율(%)")
display(payment_ratio)

['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status']
결제수단별 주문 수


payment_method
kakao_pay        79
naver_pay        77
bank_transfer    74
card             70
Name: count, dtype: int64

결제수단별 비율(%)


payment_method
kakao_pay        26.33
naver_pay        25.67
bank_transfer    24.67
card             23.33
Name: proportion, dtype: float64

##### 4) 카테고리별 평균 상품 가격

카테고리별 평균 상품 가격을 계산한 결과,  <br>
식품이 약 137143원으로 가장 높았고 생활용품이 약 96438원으로 가장 낮았다.  <br>
이는 각 카테고리에 속한 상품의 price 평균을 비교한 결과이다.

In [187]:
category_avg_price = (
    products
    .groupby("category", as_index=False)
    .agg(
        average_price=("price", "mean")
    )
)

display(category_avg_price)

,category,average_price
0,도서,106857.142857
1,뷰티,117687.500000
2,생활용품,96437.500000
3,스포츠,111578.947368
4,식품,137142.857143
5,전자기기,101588.235294
6,패션,115909.090909


##### 5) 완료 주문 금액과 전체 주문 금액의 차이

전체 주문의 line_total 합계는 255,610,000원이었고, <br>
이 중 completed 주문의 line_total 합계는 148,990,000원이었다. <br> 
두 금액의 차이는 106,620,000원으로, 이는 completed 상태가 아닌 주문에 포함된 금액이다.  <br>

In [189]:
total_amount = order_items["line_total"].sum()

completed_amount = completed_order_sales["line_total"].sum()

amount_difference = total_amount - completed_amount

print("전체 주문 금액:", total_amount)
print("완료 주문 금액:", completed_amount) #completed 상태인 주문에 포함된 line_total 합계
print("차이:", amount_difference) #완료되지 않은 주문 때문에 제외된 금액

전체 주문 금액: 255610000
완료 주문 금액: 148990000
차이: 106620000


##### 6) 고객별 구매 금액 코드를 LLM에게 요청하는 프롬프트 작성

## 정리

이번 장에서는 실제 값 확인, 컬럼 선택, 필터링, 정렬, 파생 컬럼, 병합 검증, 완료 주문 기준 집계, CSV 저장 과정을 수행했습니다. 다음 장에서는 결측치, 중복, 타입 오류, 날짜 형식, 이상값 후보를 다루는 데이터 전처리로 이어집니다.



## 0. 제출 정보
- 이름: 김혜빈
- GitHub ID: Chloe-Hyebin-Kim
- 작성일: 2026-09-29
- 최종 제출 URL: https://github.com/Chloe-Hyebin-Kim/llm-data-analysis-course/blob/main/practice/chapter04/chapter04.ipynb

## 1. 질문과 필요한 데이터 선택
### 내가 확인하려는 질문
completed 상태의 주문만을 대상으로 상품, 카테고리, 월, 고객별 구매 금액을 집계했을 때 어떤 항목의 total_sales가 높은지 확인하고, 각 집계 결과가 동일한 전체 금액을 나타내는지 검증한다.

이번 장의 total_sales는 completed 주문에 포함된 line_total의 합계이다. 회계상 순매출이라고 단정하지 않는다.

### 사용한 파일/컬럼
- orders.csv: order_id, customer_id, order_date, order_status
- order_items.csv: order_id, product_id, quantity, unit_price
- products.csv: product_id, product_name, category, price
- customers.csv: customer_id, city

### 결과 관찰
주문 금액을 계산하려면 order_items의 수량과 판매 단가가 필요하고, 완료된 주문만 선택하려면 orders의 order_status가 필요하다. 또한 상품별·카테고리별 분석을 위해 products, 고객별 분석을 위해 customers를 연결해야 한다.

### 나의 해석과 판단
분석 목적은 전체 주문 금액이 아니라 정상적으로 완료된 주문의 구매 금액을 비교하는 것이므로 order_status == "completed"를 분석 범위로 선택하였다.

line_total은 상품 한 줄의 quantity × unit_price로 계산하고, 이후 필요한 기준에 따라 카테고리·상품·월·고객 단위로 집계하였다.

불필요한 개인정보 사용을 줄이기 위해 고객별 분석에서는 이름 대신 customer_id 또는 익명화된 고객 라벨을 사용하였다.

### 업무·분석적 의미
같은 주문 데이터라도 어떤 주문 상태를 포함하는지에 따라 결과가 달라질 수 있다. 따라서 업무에서 매출 또는 구매 금액을 비교할 때는 먼저 계산 대상과 기준을 명확하게 정의해야 한다.

### 한계와 추가 확인 사항
현재 데이터만으로 할인, 쿠폰, 배송비, 세금, 부분 환불, 결제 정산 등의 정보를 모두 확인할 수 없다. 따라서 total_sales를 실제 회계상 순매출과 동일한 개념으로 해석해서는 안 된다.

In [161]:
required_files = [
    "customers.csv",
    "products.csv",
    "orders.csv",
    "order_items.csv",
]

customers = pd.read_csv(DATA_DIR / "customers.csv")
products = pd.read_csv(DATA_DIR / "products.csv")
orders = pd.read_csv(DATA_DIR / "orders.csv")
order_items = pd.read_csv(DATA_DIR / "order_items.csv")

## 2. 필터링·정렬·파생 컬럼
적용한 필터 조건: order_status == "completed"
정렬 기준: 집계 결과의 경우 total_sales 내림차순, 월별 결과는 order_month 오름차순
만든 파생 컬럼: line_total, order_month, sales_ratio, average_order_value, customer_label
line_total 계산식: quantity × unit_price


![필터와 파생 컬럼](images/step02_transform.png)

### 결과 관찰
order_items의 quantity와 unit_price를 곱해 주문 상세 한 행의 금액인 line_total을 만들었다.
주문 상태를 연결한 후에는 전체 주문 상세 중 completed 상태에 해당하는 행만 선택하여 이후 집계에 사용하였다.

전체 주문 상세 행 수: 764
completed 주문 상세 행 수: 474
completed 주문 line_total 합계: 148,990,000원
날짜 변환 실패: 0건

### 나의 해석과 판단
completed만 포함하면 취소(cancelled) 및 환불(refunded) 상태의 주문이 분석 대상에서 제외된다.

만약 모든 주문 상태를 포함하면 완료되지 않은 주문의 금액까지 합계에 포함되어 total_sales가 더 크게 계산될 수 있다. 반대로 분석 범위를 더 엄격하게 정의하면 결과가 더 작아질 수 있다.

따라서 필터 기준이 달라지면 같은 데이터에서도 분석 결과와 해석이 달라질 수 있다.

### 업무·분석적 의미
주문 상태를 구분하지 않고 금액을 합산하면 실제 완료된 거래 규모를 과대평가할 가능성이 있다. 분석 전에 주문 상태와 금액의 정의를 명확히 하는 것이 중요하다.

### 한계와 추가 확인 사항
completed라는 상태가 실제 결제 정산 완료까지 의미하는지는 현재 데이터만으로 확인할 수 없다. 실제 업무 데이터라면 주문 상태의 업무 정의와 환불 처리 기준을 추가로 확인해야 한다.


In [210]:
# STEP 2 제출용

print("===== 필터링 및 파생 컬럼 결과 확인 =====")

# 전체 주문 상세 행 수
print("전체 주문 상세 행 수:", len(order_items))

# completed 주문 상세 행 수
print("completed 주문 상세 행 수:", len(completed_order_sales))

# completed 주문 line_total 합계
print("completed 주문 line_total 합계:", completed_order_sales["line_total"].sum())

# 날짜 변환 실패
print("날짜 변환 실패:",order_sales["order_date"].isna().sum())

# completed만 필터링 제대로 한건지 확인
print("남아 있는 주문 상태:",completed_order_sales["order_status"].unique().tolist())

# line_total 계산이 정확한지 확인
line_total_check = ( order_items["line_total"]  == order_items["quantity"] * order_items["unit_price"]).all()
print("line_total 계산 일치 여부:", line_total_check)

===== 필터링 및 파생 컬럼 결과 확인 =====
전체 주문 상세 행 수: 764
completed 주문 상세 행 수: 474
completed 주문 line_total 합계: 148990000
날짜 변환 실패: 0
남아 있는 주문 상태: ['completed']
line_total 계산 일치 여부: True


## 3. merge 검증
- 병합한 데이터:
    - order_items + orders
    - completed_order_sales + products
    - customer_sales + customers
- 사용한 key:
    - 주문 병합: order_id
    - 상품 병합: product_id
    - 고객 병합: customer_id
- `validate` 결과:
    - order_items + orders: validate="many_to_one" 정상
    - completed 주문 + products: validate="many_to_one" 정상
    - 고객 집계 + customers: validate="one_to_one" 정상
- `indicator` 결과:
    - orders 병합: _merge가 모두 both
    - products 병합: _merge가 모두 both
- 병합 전/후 행 수:
    - orders merge: 764 -> 764
    - products merge: 474 -> 474
    

![merge 검증](images/step03_merge.png)


### 결과 관찰
orders.order_id 중복: 0건
orders merge 전 행 수: 764
orders merge 후 행 수: 764
orders merge 미매칭(left_only): 0건
products.product_id 중복: 0건
products merge 전 행 수: 474
products merge 후 행 수: 474
products merge 미매칭(left_only): 0건

현재 공식 샘플 데이터 기준으로 병합 전후 행 수가 유지되고 미매칭 행이 없었다.

### 나의 해석과 판단
orders.order_id와 products.product_id가 중복되지 않았고, validate="many_to_one"에서 오류가 발생하지 않았다.

또한 병합 전후 행 수가 동일하고 _merge 결과에서 left_only가 없으므로, 현재 데이터에서는 병합으로 인해 주문 상세 행이 중복되거나 누락된 증거가 없다.

따라서 이번 병합은 분석 목적에 맞게 수행되었다고 판단하였다.

### 업무·분석적 의미
잘못된 merge로 한 주문 상세가 여러 행으로 복제되면 판매 수량과 금액이 중복 집계되어 total_sales가 실제보다 크게 계산될 수 있다.

반대로 미매칭 데이터가 제거되면 실제 존재하는 주문이나 상품이 집계에서 빠질 수 있다. 따라서 집계 전에 key의 고유성, 병합 전후 행 수, 미매칭 여부를 확인해야 한다.

### 한계와 추가 확인 사항
행 수와 _merge가 정상이라는 사실만으로 모든 데이터 관계가 업무적으로 정확하다고 단정할 수는 없다. 실제 업무 데이터라면 order_id, product_id, customer_id의 참조 관계와 데이터 생성 규칙도 함께 확인해야 한다.

In [197]:
# STEP 3 제출용

print("===== 3-1. orders merge 검증 =====")
print("orders.order_id 중복:", orders["order_id"].duplicated().sum())# orders.order_id 중복 확인

# 검증용 merge
orders_merge_check = order_items.merge(
    orders[
        ["order_id", "customer_id", "order_date", "order_status"]
    ],
    on="order_id",
    how="left",
    validate="many_to_one",
    indicator=True,
)

print("orders merge 전 행 수:", len(order_items))
print("orders merge 후 행 수:", len(orders_merge_check))
print("orders merge 미매칭(left_only):",(orders_merge_check["_merge"] == "left_only").sum())
print("\n_merge 결과")
print(orders_merge_check["_merge"].value_counts())

print("\n\n===== 3-2. products merge 검증 =====")
print("products.product_id 중복:",products["product_id"].duplicated().sum())# products.product_id 중복 확인

# 검증용 merge
products_merge_check = completed_order_sales.merge(
    products,
    on="product_id",
    how="left",
    validate="many_to_one",
    indicator=True,
)

print("products merge 전 행 수:", len(completed_order_sales))
print("products merge 후 행 수:", len(products_merge_check))
print( "products merge 미매칭(left_only):",(products_merge_check["_merge"] == "left_only").sum())
print("\n_merge 결과")
print(products_merge_check["_merge"].value_counts())

===== 3-1. orders merge 검증 =====
orders.order_id 중복: 0
orders merge 전 행 수: 764
orders merge 후 행 수: 764
orders merge 미매칭(left_only): 0

_merge 결과
_merge
both          764
left_only       0
right_only      0
Name: count, dtype: int64


===== 3-2. products merge 검증 =====
products.product_id 중복: 0
products merge 전 행 수: 474
products merge 후 행 수: 474
products merge 미매칭(left_only): 0

_merge 결과
_merge
both          474
left_only       0
right_only      0
Name: count, dtype: int64


## 4. completed 주문 범위와 집계

- 분석 범위 정의: order_status == "completed"인 주문의 주문 상세
- 카테고리별 결과: category별 total_quantity, total_sales, sales_ratio
- 상품별 결과: product_id, product_name, category별 total_quantity, total_sales
- 월별 결과: order_month별 total_sales, 고유 order_count, average_order_value
- 고객별 결과: customer_id별 고유 order_count, total_sales

![핵심 집계 결과](images/step04_groupby.png)

### 결과 관찰
===== 카테고리 매출 1위 확인 =====
매출 1위 카테고리: 스포츠
판매 수량: 295
total_sales: 31743000
매출 비중(%): 21.31

===== 상품 매출 1위 확인 =====
매출 1위 상품: 스포츠 상품 041
카테고리: 스포츠
판매 수량: 35
total_sales: 5705000

===== 매출 최상위 월 확인 =====
매출이 가장 높은 월: 2025-12
total_sales: 21085000
주문 수: 23
평균 주문 금액: 916739.0

===== 매출 최하위 월 확인 =====
매출이 가장 낮은 월: 2025-09
total_sales: 2650000
주문 수: 5



### 나의 해석과 판단
가장 의미 있다고 판단한 결과는 카테고리별 매출 기여도의 차이이다.

스포츠 카테고리가 전체 completed 주문 금액의 약 21.31%로 가장 높은 비중을 차지하고, 패션은 7.11%로 가장 낮았다. 단순 판매 수량뿐 아니라 판매 단가와 상품 구성의 영향이 함께 반영되므로, 수량이 많다고 반드시 total_sales도 가장 높은 것은 아니다.

또한 특정 상품인 스포츠 상품 041의 total_sales가 가장 높았지만, 단일 상품 결과만으로 향후 수요가 계속 높을 것이라고 판단할 수는 없다.

### 업무·분석적 의미
카테고리/상품별 결과는 재고 관리, 상품 구성, 프로모션 후보 선정 등에 활용할 수 있다.

다만 현재 집계는 과거 completed 주문을 요약한 것이므로 실제 발주량을 결정하려면 재고 수준이나 기간별 수요 변화, 원자재 가격 변화 등의 시장 상황, 공급 기간 등의 추가 정보가 필요하다.

월별 결과는 구매 금액의 시간적 변화가 있는지 확인하는 다음 분석으로 연결할 수 있고, 고객별 결과는 반복 구매 고객이나 재구매 분석을 통해 상품의 만족도를 파악하거나 고객군 분석으로 확장할 수 있다.

### 한계와 추가 확인 사항
otal_sales는 quantity × unit_price의 합계일 뿐이기 떄문이다. 따라서 부분 취소 및 부분 환불이나 쿠폰사용 및 결제 수수료나 할부 그리고 전산금액, 세금 등을 반영한 회계상 순매출인지 확인할 수 없다.
이러한 이유로 현재 결과를 회계상 순매출과 같다고 단정하지 않는다.

In [198]:
# STEP 4 제출용

print("===== 카테고리별 결과 내림차순 정렬 =====")

display(
    category_sales[
        [
            "category",
            "total_quantity",
            "total_sales",
            "sales_ratio"
        ]
    ]
)

===== 카테고리별 결과 내림차순 정렬 =====


,category,total_quantity,total_sales,sales_ratio
3,스포츠,295,31743000,21.31
5,전자기기,259,26400000,17.72
2,생활용품,272,23915000,16.05
1,뷰티,223,23383000,15.69
4,식품,133,16573000,11.12
0,도서,149,16389000,11.00
6,패션,111,10587000,7.11


In [203]:
# STEP 4 제출용

print("===== 상품별 결과 =====")

display(
    product_sales[
        [
            "product_id",
            "product_name",
            "category",
            "total_quantity",
            "total_sales"
        ]
    ].head(10)
)

===== 상품별 결과 =====


,product_id,product_name,category,total_quantity,total_sales
39,41,스포츠 상품 041,스포츠,35,5705000
11,12,식품 상품 012,식품,25,4375000
8,9,스포츠 상품 009,스포츠,20,3860000
70,72,뷰티 상품 072,뷰티,20,3780000
69,71,전자기기 상품 071,전자기기,23,3703000
66,68,스포츠 상품 068,스포츠,26,3640000
78,81,전자기기 상품 081,전자기기,22,3630000
10,11,패션 상품 011,패션,31,3565000
20,22,생활용품 상품 022,생활용품,29,3248000
86,89,생활용품 상품 089,생활용품,30,3090000


In [204]:
print("===== 월별 결과 =====")

display(
    monthly_summary[
        [
            "order_month",
            "total_sales",
            "order_count",
            "average_order_value"
        ]
    ]
)

===== 월별 결과 =====


,order_month,total_sales,order_count,average_order_value
0,2025-09,2650000,5,530000.0
1,2025-10,16970000,19,893158.0
2,2025-11,11021000,13,847769.0
3,2025-12,21085000,23,916739.0
4,2026-01,14272000,16,892000.0
5,2026-02,4784000,7,683429.0
6,2026-03,18323000,25,732920.0
7,2026-04,11546000,16,721625.0
8,2026-05,16631000,19,875316.0
9,2026-06,12504000,17,735529.0


In [205]:
# STEP 4 고객별 제출용

customer_sales = (
    completed_order_sales
    .groupby("customer_id", as_index=False)
    .agg(
        order_count=("order_id", "nunique"),
        total_sales=("line_total", "sum"),
    )
    .sort_values("total_sales",ascending=False)
)

# 고객의 도시 정보 연결
customer_sales = customer_sales.merge(
    customers[["customer_id", "city"]],
    on="customer_id",
    how="left",
    validate="one_to_one",
)

print("===== 고객별 결과 =====")

display(
    customer_sales[
        [
            "customer_id",
            "city",
            "order_count",
            "total_sales"
        ]
    ].head(10)
)

===== 고객별 결과 =====


,customer_id,city,order_count,total_sales
0,117,성남,5,4100000
1,102,고양,4,3996000
2,83,수원,4,3880000
3,30,서울,5,3590000
4,40,서울,4,3523000
5,20,인천,2,3191000
6,3,성남,2,3178000
7,111,광주,3,3153000
8,66,서울,4,3093000
9,147,부산,2,2990000


In [212]:
# STEP 4 제출용

print("\n===== 카테고리 매출 1위 확인 =====")
top_category = category_sales.iloc[0]
print("매출 1위 카테고리:", top_category["category"])
print("판매 수량:", top_category["total_quantity"])
print("total_sales:", top_category["total_sales"])
print("매출 비중(%):", top_category["sales_ratio"])

print("\n===== 상품 매출 1위 확인 =====")
top_product = product_sales.iloc[0]
print("매출 1위 상품:", top_product["product_name"])
print("카테고리:", top_product["category"])
print("판매 수량:", top_product["total_quantity"])
print("total_sales:", top_product["total_sales"])

print("\n===== 매출 최상위 월 확인 =====")
top_month = monthly_summary.loc[monthly_summary["total_sales"].idxmax()]
print("매출이 가장 높은 월:", top_month["order_month"])
print("total_sales:", top_month["total_sales"])
print("주문 수:", top_month["order_count"])
print("평균 주문 금액:", top_month["average_order_value"])


print("\n===== 매출 최하위 월 확인 =====")
bottom_month = monthly_summary.loc[monthly_summary["total_sales"].idxmin()]
print("매출이 가장 낮은 월:", bottom_month["order_month"])
print("total_sales:", bottom_month["total_sales"])
print("주문 수:", bottom_month["order_count"])


print("\n===== 고객별 total_sales 1위 =====")
top_customer = customer_sales.iloc[0]
print("customer_id:", top_customer["customer_id"])
print("도시:", top_customer["city"])
print("완료 주문 수:", top_customer["order_count"])
print("total_sales:", top_customer["total_sales"])


===== 카테고리 매출 1위 확인 =====
매출 1위 카테고리: 스포츠
판매 수량: 295
total_sales: 31743000
매출 비중(%): 21.31

===== 상품 매출 1위 확인 =====
매출 1위 상품: 스포츠 상품 041
카테고리: 스포츠
판매 수량: 35
total_sales: 5705000

===== 매출 최상위 월 확인 =====
매출이 가장 높은 월: 2025-12
total_sales: 21085000
주문 수: 23
평균 주문 금액: 916739.0

===== 매출 최하위 월 확인 =====
매출이 가장 낮은 월: 2025-09
total_sales: 2650000
주문 수: 5

===== 고객별 total_sales 1위 =====
customer_id: 117
도시: 성남
완료 주문 수: 5
total_sales: 4100000


## 5. 총합 일치 검증
- 원본 completed line_total 합계: 148,990,000원
- 카테고리 합계: 148,990,000원
- 월별 합계: 148,990,000원
- 고객별 합계: 148,990,000원
- 차이 여부: PASS — 모두 일치


![총합 검증](images/step05_total_check.png)


### 나의 해석과 판단
같은 completed 주문 데이터를 서로 다른 기준으로 집계했더라도 전체  `total_sales`는 동일해야 한다.

카테고리/월별/고객별 합계가 원본 completed 주문 상세의 line_total 합계와 모두 같다면, 집계 과정에서 금액이 누락되거나 중복되지 않았다는 것을 확인하는 하나의 검증 근거가 된다.

만약 총합이 일치하지 않는다면 다음 순서로 확인한다.

모든 집계가 동일하게 completed 주문만 사용했는지 확인한다.
merge 전후 행 수가 달라졌는지 확인한다.
_merge에서 left_only 또는 미매칭 데이터가 있는지 확인한다.
merge key에 중복이 있어 행이 복제되지 않았는지 확인한다.
order_month, category, customer_id 등에 결측치가 있어 groupby() 과정에서 제외된 행이 있는지 확인한다.
모든 집계에서 동일한 line_total 계산식을 사용했는지 확인한다.

총합 검증은 코드가 오류 없이 실행됐는지가 아니라 서로 다른 분석 결과가 동일한 원본 금액을 보존하는지 확인하는 과정이라고 판단하였다.

In [218]:
# STEP 5 제출용

print("===== 5. 총합 일치 검증 =====")

# total_sales 합계
completed_total = completed_order_sales["line_total"].sum()
category_total = category_sales["total_sales"].sum()
monthly_total = monthly_summary["total_sales"].sum()
customer_total = customer_sales["total_sales"].sum()

print("원본 completed line_total 합계:", completed_total)
print("카테고리 합계:", category_total)
print("월별 합계:", monthly_total)
print("고객별 합계:", customer_total)

print("\n===== 차이 확인 =====")
print("카테고리 차이:", category_total - completed_total)
print("월별 차이:", monthly_total - completed_total)
print("고객별 차이:", customer_total - completed_total)

total_check = (completed_total == category_total == monthly_total == customer_total)
print("\n합계 일치 여부:","PASS" if total_check else "FAIL")

===== 5. 총합 일치 검증 =====
원본 completed line_total 합계: 148990000
카테고리 합계: 148990000
월별 합계: 148990000
고객별 합계: 148990000

===== 차이 확인 =====
카테고리 차이: 0
월별 차이: 0
고객별 차이: 0

합계 일치 여부: PASS



## 6. LLM pandas 코드 검증
- LLM Prompt 요약:
```text
orders와 order_items를 사용해
completed 주문 기준 월별 금액을 계산하려고 합니다.

다음을 반드시 포함해 주세요.

- line_total = quantity × unit_price
- orders.order_id 고유성 확인
- many-to-one merge + indicator
- 병합 전후 행 수 확인
- 미매칭 확인
- 날짜 변환 실패 확인
- completed만 필터링
- 월별 total_sales와 고유 order_count 계산
- 결과를 회계상 순매출이라고 단정하지 않기
```
- 제안 코드 요약:
```python
# 1. orders.order_id 고유성 확인
print(
    "orders.order_id 중복 수:",
    orders["order_id"].duplicated().sum()
)


# 2. line_total 생성
order_items_check = order_items.copy()

order_items_check["line_total"] = (
    order_items_check["quantity"]
    * order_items_check["unit_price"]
)


# 3. orders와 merge
monthly_check = order_items_check.merge(
    orders[
        [
            "order_id",
            "order_date",
            "order_status"
        ]
    ],
    on="order_id",
    how="left",
    validate="many_to_one",
    indicator=True
)


# 4. 병합 전후 행 수 확인
print("병합 전 행 수:", len(order_items_check))
print("병합 후 행 수:", len(monthly_check))


# 5. 미매칭 확인
print(
    "미매칭(left_only):",
    (monthly_check["_merge"] == "left_only").sum()
)

print(monthly_check["_merge"].value_counts())


# 6. 날짜 변환
monthly_check["order_date"] = pd.to_datetime(
    monthly_check["order_date"],
    errors="coerce"
)

print(
    "날짜 변환 실패:",
    monthly_check["order_date"].isna().sum()
)


# 7. completed 주문만 필터링
completed_check = monthly_check[
    monthly_check["order_status"] == "completed"
].copy()


# 8. 월 컬럼 생성
completed_check["order_month"] = (
    completed_check["order_date"]
    .dt.to_period("M")
    .astype(str)
)


# 9. 월별 집계
llm_monthly_summary = (
    completed_check
    .groupby("order_month", as_index=False)
    .agg(
        total_sales=("line_total", "sum"),
        order_count=("order_id", "nunique")
    )
    .sort_values("order_month")
)

display(llm_monthly_summary)
```
  quantity × unit_price로 line_total을 만든 뒤, order_items와 orders를 order_id 기준으로 병합하고,
  completed 주문만 필터링하여 월별 total_sales와 order_count를 계산하였다.

- 실제 컬럼/범위와 맞지 않은 부분:   요청한 주요 데이터 검증 항목은 모두 포함되어 있었다.
  다만 계산된 total_sales가 회계상 순매출과 동일하다고 단정할 수 없다는 결과 해석상의 주의 문구는 코드에 포함되지 않았다.
- 수정한 내용:  total_sales는 completed 주문의 quantity × unit_price 합계이며,
  할인, 환불 세부 내역, 배송비, 세금 등의 정보가 반영된 회계상 순매출이라고 단정할 수 없다는 설명을 추가하였다.
- 최종 판단: 수정 후 사용

![LLM 코드 검증](images/step06_llm_validation.png)

### 나의 해석과 판단
LLM 코드가 실행되었다는 것만으로 분석이 맞다고 볼 수 없는 이유를 작성하세요.


In [216]:
# 1. orders.order_id 고유성 확인
print(
    "orders.order_id 중복 수:",
    orders["order_id"].duplicated().sum()
)


# 2. line_total 생성
order_items_check = order_items.copy()

order_items_check["line_total"] = (
    order_items_check["quantity"]
    * order_items_check["unit_price"]
)


# 3. orders와 merge
monthly_check = order_items_check.merge(
    orders[
        [
            "order_id",
            "order_date",
            "order_status"
        ]
    ],
    on="order_id",
    how="left",
    validate="many_to_one",
    indicator=True
)


# 4. 병합 전후 행 수 확인
print("병합 전 행 수:", len(order_items_check))
print("병합 후 행 수:", len(monthly_check))


# 5. 미매칭 확인
print(
    "미매칭(left_only):",
    (monthly_check["_merge"] == "left_only").sum()
)

print(monthly_check["_merge"].value_counts())


# 6. 날짜 변환
monthly_check["order_date"] = pd.to_datetime(
    monthly_check["order_date"],
    errors="coerce"
)

print(
    "날짜 변환 실패:",
    monthly_check["order_date"].isna().sum()
)


# 7. completed 주문만 필터링
completed_check = monthly_check[
    monthly_check["order_status"] == "completed"
].copy()


# 8. 월 컬럼 생성
completed_check["order_month"] = (
    completed_check["order_date"]
    .dt.to_period("M")
    .astype(str)
)


# 9. 월별 집계
llm_monthly_summary = (
    completed_check
    .groupby("order_month", as_index=False)
    .agg(
        total_sales=("line_total", "sum"),
        order_count=("order_id", "nunique")
    )
    .sort_values("order_month")
)

display(llm_monthly_summary)

orders.order_id 중복 수: 0
병합 전 행 수: 764
병합 후 행 수: 764
미매칭(left_only): 0
_merge
both          764
left_only       0
right_only      0
Name: count, dtype: int64
날짜 변환 실패: 0


,order_month,total_sales,order_count
0,2025-09,2650000,5
1,2025-10,16970000,19
2,2025-11,11021000,13
3,2025-12,21085000,23
4,2026-01,14272000,16
5,2026-02,4784000,7
6,2026-03,18323000,25
7,2026-04,11546000,16
8,2026-05,16631000,19
9,2026-06,12504000,17


In [217]:
print("===== LLM 코드 검증 =====")

print(
    "실제 컬럼명 사용:",
    {
        "order_id",
        "quantity",
        "unit_price"
    }.issubset(order_items.columns)
)

print(
    "completed만 존재:",
    completed_check["order_status"].unique().tolist()
)

print(
    "월별 total_sales 합계:",
    llm_monthly_summary["total_sales"].sum()
)

print(
    "completed line_total 합계:",
    completed_check["line_total"].sum()
)

print(
    "합계 일치 여부:",
    llm_monthly_summary["total_sales"].sum()
    == completed_check["line_total"].sum()
)

===== LLM 코드 검증 =====
실제 컬럼명 사용: True
completed만 존재: ['completed']
월별 total_sales 합계: 148990000
completed line_total 합계: 148990000
합계 일치 여부: True



## 7. Chapter 04 최종 인사이트
### 가장 의미 있다고 생각한 결과 2가지
1. completed 주문 기준 카테고리별 total_sales를 비교한 결과, 스포츠 카테고리가 31,743,000원으로 가장 높았으며 전체 completed 주문 금액의 약 21.31%를 차지하였다.

2. 상품별·카테고리별·월별·고객별로 서로 다른 기준으로 집계했지만, 각 total_sales 합계가 모두 148,990,000원으로 일치하였다.
   이를 통해 집계 과정에서 금액이 누락되거나 중복되지 않았음을 확인하였다.

### 그 결과를 뒷받침하는 수치/표
- completed 주문 상세 line_total 합계: 148,990,000원
- category_sales 합계: 148,990,000원
- monthly_summary 합계: 148,990,000원
- customer_sales 합계: 148,990,000원
- 합계 일치 여부: PASS

카테고리별 결과에서는 스포츠가 31,743,000원으로 가장 높았고,
상품별 결과에서는 스포츠 상품 041이 5,705,000원으로 가장 높았다.


### 추가로 확인하고 싶은 질문
스포츠 카테고리의 total_sales가 높은 이유가 판매 수량이 많기 때문인지, 상품의 판매 단가가 높기 때문인지 추가로 비교해 보고 싶다.
또한 월별 total_sales 변화가 주문 수의 변화 때문인지,주문당 구매 금액의 차이 때문인지도 추가로 확인하고 싶다.

### 현재 결과의 한계
이번 분석의 total_sales는 completed 주문에 포함된 quantity × unit_price의 합계이다.
할인, 쿠폰, 배송비, 세금, 부분 환불, 실제 결제 및 정산 금액 등의 정보를 모두 반영한 값인지는 현재 데이터만으로 확인할 수 없다.
따라서 이번 결과를 회계상 순매출과 동일하다고 단정할 수 없다.

## 최종 제출 체크
- [O] 핵심 셀 Output이 남아 있습니다.
- [O] merge와 총합 검증 Evidence가 있습니다.
- [O] 결과 관찰과 해석이 구분되어 있습니다.
- [O] LLM 코드를 검증했습니다.
- [O] 개인정보/Secret이 없습니다.
- [O] `chapter04/chapter04.ipynb`가 GitHub에서 정상 표시됩니다.
- [O] 최종 Notebook 파일 URL을 제출합니다.

[Chapter 04 Evidence]

1. Notebook
- notebooks/ch04_pandas_basic.ipynb 실행 완료: [예]

2. 병합 검증
- orders.order_id 중복: 0
- orders merge 전 행 수: 764
- orders merge 후 행 수: 764
- orders merge 미매칭: 0
- products merge 전 행 수: 474
- products merge 후 행 수: 474
- products merge 미매칭: 0

3. 계산 범위
- 사용 주문 상태: completed
- line_total 계산식: quantity × unit_price
- 날짜 변환 실패: 0

4. 교차 검증
- 완료 주문 상세 line_total 합계: 148,990,000
- category_sales 합계: 148,990,000
- monthly_summary 합계: 148,990,000
- customer_sales 합계: 148,990,000
- 합계 일치 여부: PASS

5. 저장 파일
- ch04_category_sales.csv: [존재]
- ch04_product_sales.csv: [존재]
- ch04_monthly_sales.csv: [존재]
- ch04_customer_sales.csv: [존재]

6. LLM 검증
- LLM이 빠뜨린 검증 항목: 실제 컬럼 확인, completed 필터링, 미매칭 확인, 총합 교차 검증

7. 남은 질문
- 스포츠 카테고리의 높은 total_sales가 판매 수량과 상품 단가 중 어느 요인의 영향을 더 많이 받았는지 확인하고 싶다.

In [213]:
print("===================================")
print("Chapter 04 Evidence 검증")
print("===================================")


# =========================================================
# 2. 병합 검증 - orders
# =========================================================

print("\n[2-1. orders merge 검증]")

# orders.order_id 중복 확인
orders_duplicate = orders["order_id"].duplicated().sum()

# 검증용 merge
orders_merge_check = order_items.merge(
    orders[
        [
            "order_id",
            "customer_id",
            "order_date",
            "order_status"
        ]
    ],
    on="order_id",
    how="left",
    validate="many_to_one",
    indicator="orders_merge_status"
)

orders_before = len(order_items)
orders_after = len(orders_merge_check)

orders_unmatched = (
    orders_merge_check["orders_merge_status"] == "left_only"
).sum()


print("orders.order_id 중복:", orders_duplicate)
print("orders merge 전 행 수:", orders_before)
print("orders merge 후 행 수:", orders_after)
print("orders merge 미매칭:", orders_unmatched)


# =========================================================
# 2. 병합 검증 - products
# =========================================================

print("\n[2-2. products merge 검증]")

# products.product_id 중복 확인
products_duplicate = products["product_id"].duplicated().sum()

# completed 주문만 선택
completed_merge_check = orders_merge_check[
    orders_merge_check["order_status"] == "completed"
].copy()

# products 병합
products_merge_check = completed_merge_check.merge(
    products,
    on="product_id",
    how="left",
    validate="many_to_one",
    indicator="products_merge_status"
)

products_before = len(completed_merge_check)
products_after = len(products_merge_check)

products_unmatched = (
    products_merge_check["products_merge_status"] == "left_only"
).sum()


print("products.product_id 중복:", products_duplicate)
print("products merge 전 행 수:", products_before)
print("products merge 후 행 수:", products_after)
print("products merge 미매칭:", products_unmatched)


# =========================================================
# 3. 계산 범위
# =========================================================

print("\n[3. 계산 범위]")

# 날짜 변환
date_check = pd.to_datetime(
    orders_merge_check["order_date"],
    errors="coerce"
)

date_fail = date_check.isna().sum()


print("사용 주문 상태: completed")
print("line_total 계산식: quantity × unit_price")
print("날짜 변환 실패:", date_fail)


# =========================================================
# 4. 교차 검증
# =========================================================

print("\n[4. 교차 검증]")

completed_total = completed_order_sales["line_total"].sum()
category_total = category_sales["total_sales"].sum()
monthly_total = monthly_summary["total_sales"].sum()
customer_total = customer_sales["total_sales"].sum()


print(
    "완료 주문 상세 line_total 합계:",
    f"{completed_total:,.0f}"
)

print(
    "category_sales 합계:",
    f"{category_total:,.0f}"
)

print(
    "monthly_summary 합계:",
    f"{monthly_total:,.0f}"
)

print(
    "customer_sales 합계:",
    f"{customer_total:,.0f}"
)


# 모든 합계가 같은지 확인
total_pass = (
    completed_total
    == category_total
    == monthly_total
    == customer_total
)

print(
    "합계 일치 여부:",
    "PASS" if total_pass else "FAIL"
)

Chapter 04 Evidence 검증

[2-1. orders merge 검증]
orders.order_id 중복: 0
orders merge 전 행 수: 764
orders merge 후 행 수: 764
orders merge 미매칭: 0

[2-2. products merge 검증]
products.product_id 중복: 0
products merge 전 행 수: 474
products merge 후 행 수: 474
products merge 미매칭: 0

[3. 계산 범위]
사용 주문 상태: completed
line_total 계산식: quantity × unit_price
날짜 변환 실패: 0

[4. 교차 검증]
완료 주문 상세 line_total 합계: 148,990,000
category_sales 합계: 148,990,000
monthly_summary 합계: 148,990,000
customer_sales 합계: 148,990,000
합계 일치 여부: PASS
